<a href="https://colab.research.google.com/github/sabrinareyes/fake-news-app/blob/main/step1_data_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fake News Detection — Step 1: Data Preparation

**Dataset:** LIAR (Wang, *"Liar, Liar Pants on Fire"*, ACL 2017) — ~12.8K short political
statements from PolitiFact, each hand-labeled with one of six truthfulness ratings.

**Method:** following Galli et al., *"A comprehensive Benchmark for fake news detection"*
(J Intell Inf Syst, 2022) — classical ML classifiers on TF-IDF text features.

This notebook covers Step 1 only: **load → relabel → clean**. The output is three CSVs
that Step 2 (TF-IDF + Logistic Regression) reads directly.

## 0. Setup

Only `pandas` is needed for this step. `re` and `html` are in the standard library.

In [ ]:
import html
import re
import urllib.request
import zipfile
from pathlib import Path

import pandas as pd

DATA = Path("data")
DATA.mkdir(exist_ok=True)

## 1. Get the raw files

The original release is a zip of three TSVs from the author's page at UCSB. Run this once;
it skips the download if the files are already there.

If the UCSB link is unreachable from your network, the same three TSVs are mirrored in
several public GitHub repos — but cite the UCSB original in your write-up, since that is
the primary source.

In [ ]:
URL = "https://www.cs.ucsb.edu/~william/data/liar_dataset.zip"

if not (DATA / "train.tsv").exists():
    zip_path = DATA / "liar_dataset.zip"
    urllib.request.urlretrieve(URL, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA)
    print("downloaded and extracted")
else:
    print("files already present")

sorted(p.name for p in DATA.glob("*.tsv"))

downloaded and extracted


['test.tsv', 'train.tsv', 'valid.tsv']

## 2. Load the TSVs

The files have **no header row**, so we supply the 14 column names from the dataset README.

Two choices worth understanding:

- **`quoting=3` (`QUOTE_NONE`)** — statements contain unbalanced double quotes, e.g.
  `Says "X" about Y`. With pandas' default quote handling, an opening quote makes the parser
  swallow everything up to the next quote, silently merging fields or dropping rows. This
  file is tab-delimited and nothing else, so we tell pandas exactly that.
- **`keep_default_na=False`** — otherwise a statement consisting of the literal word `"null"`
  or an empty context field becomes `NaN`, and we would have to guard every string operation.

### About columns 8–12 (the credit-history counts)

These are the speaker's career tallies of each rating. The dataset README states they
**include the current statement** — so a statement labeled `false` has already been counted
in that speaker's `false_ct`. Feeding them to a classifier leaks the answer and produces
inflated accuracy that will not survive scrutiny.

**We ignore them entirely and model the statement text only**, which is also what the
benchmark paper does. Worth a sentence in your write-up: it is a well-known trap with LIAR.

In [ ]:
COLUMNS = [
    "id", "label", "statement", "subjects", "speaker", "job_title",
    "state_info", "party", "barely_true_ct", "false_ct", "half_true_ct",
    "mostly_true_ct", "pants_fire_ct", "context",
]


def load_split(name):
    """Read one raw LIAR .tsv into a DataFrame."""
    return pd.read_csv(
        DATA / f"{name}.tsv",
        sep="\t",
        header=None,
        names=COLUMNS,
        quoting=3,             # QUOTE_NONE
        dtype=str,
        keep_default_na=False,
    )


raw_train = load_split("train")
print(raw_train.shape)
raw_train.head(3)

(10269, 14)


,id,label,statement,subjects,speaker,job_title,state_info,party,barely_true_ct,false_ct,half_true_ct,mostly_true_ct,pants_fire_ct,context
0,2635.json,false,Says the Annies List political group supports ...,abortion,dwayne-bohac,State representative,Texas,republican,0,1,0,0,0,a mailer
1,10540.json,half-true,When did the decline of coal start? It started...,"energy,history,job-accomplishments",scott-surovell,State delegate,Virginia,democrat,0,0,1,1,0,a floor speech.
2,324.json,mostly-true,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy,barack-obama,President,Illinois,democrat,70,71,160,163,9,Denver


### Sanity check the row counts

The published split sizes are **10,269 / 1,284 / 1,283** (12,836 total). If your numbers
differ, the parser is mangling rows — most likely the quoting setting.

In [ ]:
for name, expected in [("train", 10269), ("valid", 1284), ("test", 1283)]:
    n = len(load_split(name))
    print(f"{name:<6} {n:>6} rows   expected {expected:>6}   {'OK' if n == expected else 'MISMATCH'}")

train   10269 rows   expected  10269   OK
valid    1284 rows   expected   1284   OK
test     1283 rows   expected   1283   OK


## 3. Collapse six labels into two

PolitiFact's six ratings, ordered from most to least deceptive:

`pants-fire` → `false` → `barely-true` → `half-true` → `mostly-true` → `true`

We cut this ordinal scale into a binary one. The cut point is a judgment call, so we define
**two** mappings and carry both through — the second is a sensitivity check for the write-up.

| mapping | fake (1) | real (0) |
|---|---|---|
| **primary (3/3)** | pants-fire, false, barely-true | half-true, mostly-true, true |
| alternative (4/2) | + half-true | mostly-true, true |

**Why the 3/3 split is the primary one:** it is what most LIAR binary papers use (so your
accuracy is comparable to published numbers), and it keeps the classes near-balanced. The
4/2 split pushes the majority class to ~64%, which means a model scoring 62% would be
*worse than always guessing "fake"* — a bad look in a demo, and not a fair test of the model.

**The honest caveat:** `barely-true` is the weakest link. PolitiFact defines it as
"contains an element of truth but ignores critical facts" — that is spin more than
fabrication. Calling it fake is defensible, but say so rather than presenting the boundary
as self-evident.

In [ ]:
LABEL_MAP = {
    "pants-fire":  1,
    "false":       1,
    "barely-true": 1,
    "half-true":   0,
    "mostly-true": 0,
    "true":        0,
}

# Alternative: half-true also counts as deceptive.
LABEL_MAP_STRICT = {**LABEL_MAP, "half-true": 1}

## 4. Clean the text

Deliberately conservative. Each operation earns its place:

| step | why |
|---|---|
| `html.unescape` | a few rows carry `&amp;`, `&quot;` from scraping |
| strip `<tags>` | stray HTML would become junk tokens |
| strip URLs | a URL appears once and never generalizes — pure noise to TF-IDF |
| lowercase | TF-IDF is case-sensitive; `Says` and `says` would be separate features |
| normalize curly quotes | `don't` and `don't` would otherwise be different tokens |
| collapse whitespace | tidies the tokenizer input |

### What we deliberately do NOT do

**No stopword removal and no stemming.** This is the opposite of the usual textbook advice,
so be ready to defend it: TF-IDF's *idf* term already down-weights words that appear
everywhere, and on LIAR the function words carry real signal — hedges and absolutes like
*"never"*, *"all"*, *"says"* are among the more predictive tokens. Stripping them typically
costs you a point or two. If you want, make it an ablation in your write-up.

In [ ]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
TAG_RE = re.compile(r"<[^>]+>")
WS_RE = re.compile(r"\s+")


def clean_text(s):
    s = html.unescape(s)
    s = TAG_RE.sub(" ", s)
    s = URL_RE.sub(" ", s)
    s = s.lower()
    s = s.replace("’", "'").replace("“", '"').replace("”", '"')
    s = WS_RE.sub(" ", s)
    return s.strip()


demo = 'Says <b>Barack Obama</b> &quot;never&quot; visited   Texas. See https://t.co/abc123'
print("before:", demo)
print("after :", clean_text(demo))

before: Says <b>Barack Obama</b> &quot;never&quot; visited   Texas. See https://t.co/abc123
after : says barack obama "never" visited texas. see


## 5. Handle short and empty statements

A handful of rows are truncated fragments where the actual claim did not survive scraping.
They are not merely useless — they are **actively harmful**, because the same stub appears
with contradictory labels. No model can learn anything from these, and they add noise to
both training and evaluation.

Run the cell below to see them. This is a good concrete example for a write-up section on
data quality.

In [ ]:
MIN_WORDS = 3

stubs = load_split("train")
stubs["clean"] = stubs["statement"].map(clean_text)
stubs = stubs[stubs["clean"].str.split().str.len() < MIN_WORDS]
stubs[["label", "statement"]]

,label,statement
709,false,On abortion
1014,half-true,On abortion
1071,true,On torture.
2953,false,On reconciliation
6151,false,Were bankrupt.
6548,false,On abortion.
6780,half-true,On torture.
7667,false,On sequestration


## 6. Put it together

`prepare()` runs the whole pipeline for one split and reports what it dropped.

In [ ]:
def prepare(name):
    df = load_split(name)
    n0 = len(df)

    # Drop rows whose label is not one of the six known ratings (malformed lines).
    df = df[df["label"].isin(LABEL_MAP)].copy()
    n_bad = n0 - len(df)

    df = df.rename(columns={"label": "label_raw"})
    df["statement"] = df["statement"].map(clean_text)

    too_short = df["statement"].str.split().str.len() < MIN_WORDS
    n_short = int(too_short.sum())
    df = df[~too_short].copy()

    df["label"] = df["label_raw"].map(LABEL_MAP)
    df["label_strict"] = df["label_raw"].map(LABEL_MAP_STRICT)

    print(f"[{name}] {n0} rows -> {len(df)} kept "
          f"(dropped {n_bad} malformed, {n_short} under {MIN_WORDS} words)")
    return df[["statement", "label_raw", "label", "label_strict"]]


splits = {name: prepare(name) for name in ("train", "valid", "test")}
splits["train"].head()

[train] 10269 rows -> 10261 kept (dropped 0 malformed, 8 under 3 words)
[valid] 1284 rows -> 1284 kept (dropped 0 malformed, 0 under 3 words)
[test] 1283 rows -> 1281 kept (dropped 0 malformed, 2 under 3 words)


,statement,label_raw,label,label_strict
0,says the annies list political group supports ...,false,1,1
1,when did the decline of coal start? it started...,half-true,0,1
2,"hillary clinton agrees with john mccain ""by vo...",mostly-true,0,0
3,health care reform legislation is likely to ma...,false,1,1
4,the economic turnaround started at the end of ...,half-true,0,1


## 7. Diagnostics

Everything below is material for the write-up.

### 7a. Six-class distribution

In [ ]:
pd.DataFrame(
    {k: v["label_raw"].value_counts() for k, v in splits.items()}
).fillna(0).astype(int)

,train,valid,test
label_raw,,,
barely-true,1657,237,214
false,1993,263,248
half-true,2121,248,267
mostly-true,1966,251,249
pants-fire,842,116,92
true,1682,169,211


### 7b. Binary balance and the majority-class baseline

**The most important number in this notebook.** The majority-class baseline is what you
score by ignoring the text entirely and always predicting the more common class. Any
accuracy you report has to be judged against it — that is the difference between "my model
gets 60%" and "my model beats the trivial baseline by 4 points."

In [ ]:
rows = []
for k, v in splits.items():
    for name, col in [("primary 3/3", "label"), ("alt 4/2", "label_strict")]:
        fake = v[col].mean()
        rows.append({
            "split": k, "mapping": name, "n": len(v),
            "fake %": round(100 * fake, 1),
            "real %": round(100 * (1 - fake), 1),
            "majority baseline %": round(100 * max(fake, 1 - fake), 1),
        })
pd.DataFrame(rows)

,split,mapping,n,fake %,real %,majority baseline %
0,train,primary 3/3,10261,43.8,56.2,56.2
1,train,alt 4/2,10261,64.4,35.6,64.4
2,valid,primary 3/3,1284,48.0,52.0,52.0
3,valid,alt 4/2,1284,67.3,32.7,67.3
4,test,primary 3/3,1281,43.2,56.8,56.8
5,test,alt 4/2,1281,64.1,35.9,64.1


### 7c. Statement length

Note the median: **~17 words.** That is a single sentence, not an article. This is the
structural reason LIAR accuracy sits near chance-plus-a-bit no matter which classifier you
use — there is simply very little text per example for TF-IDF to work with. It is also why
BERT helps in Step 3, but not dramatically.

In [ ]:
pd.DataFrame({
    k: v["statement"].str.split().str.len().describe()[["mean", "50%", "min", "max"]]
    for k, v in splits.items()
}).round(1)

,train,valid,test
mean,17.9,17.9,18.0
50%,17.0,17.0,16.0
min,3.0,3.0,3.0
max,66.0,57.0,48.0


### 7d. Leakage check

Do any test statements appear verbatim in the training set? A few short stubs do. At ~0.3%
it will not move your numbers, but checking (and saying you checked) is the kind of rigor
that gets noticed.

In [ ]:
train_set = set(splits["train"]["statement"])
for k in ("valid", "test"):
    dup = splits[k]["statement"].isin(train_set)
    print(f"{k}: {dup.sum()} of {len(splits[k])} statements also appear in train")

splits["test"][splits["test"]["statement"].isin(train_set)]

valid: 5 of 1284 statements also appear in train
test: 4 of 1281 statements also appear in train


,statement,label_raw,label,label_strict
94,says milken institute rated san antonio as nat...,true,0,0
598,on common core.,false,1,1
1185,on a cap-and-trade plan.,false,1,1
1274,on offshore drilling.,half-true,0,1


## 8. Save

Step 2 reads these three files.

In [ ]:
for name, df in splits.items():
    path = DATA / f"{name}_clean.csv"
    df.to_csv(path, index=False)
    print(f"wrote {path}  ({len(df)} rows)")

wrote data/train_clean.csv  (10261 rows)
wrote data/valid_clean.csv  (1284 rows)
wrote data/test_clean.csv  (1281 rows)


---

## Limitations to state honestly in your write-up

1. **The headline number is modest by construction.** The published LIAR benchmarks land
   around 60% binary accuracy against a ~57% majority baseline. Report both numbers
   together — the model contributes roughly 3–4 points, and framing it that way is more
   credible than quoting 60% alone.
2. **Statements are ~17 words.** TF-IDF over a single sentence is a very thin feature
   vector. This caps performance regardless of classifier choice.
3. **US politics, 2007–2016.** Speakers, topics, and phrasing are narrow. The model will not
   transfer to other domains, other countries, or later events — do not claim it detects
   "fake news" in general.
4. **PolitiFact's ratings are editorial judgments**, not ground truth. Sampling is not
   random either: PolitiFact fact-checks claims it finds *interesting*, which skews toward
   the contentious.
5. **The binary cut point is a design decision**, not a property of the data — hence the
   two mappings above.
6. **Credit-history columns are excluded** because they leak the label. Papers reporting
   very high LIAR accuracy have sometimes used them without noticing.

## Next: Step 2

TF-IDF vectorization → Logistic Regression baseline → accuracy / precision / recall / F1 /
confusion matrix → comparison against Linear SVM and Naive Bayes.